# Unit Tests — Weakness Detection

A single, self-contained test notebook for the Offensive IT-Tester. Each section asks **one
key question** about a layer — the most important weakness that layer could have — and checks
it against the live system. The notebook starts the sandbox itself; just **Run All**.

## 0. Setup — imports, sandbox, and a tiny test harness

In [1]:
import sys, threading, time, json, tempfile, urllib.parse
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "config" / "paths.py").exists():
        if str(_p) not in sys.path: sys.path.insert(0, str(_p))
        ROOT = _p; break

import pandas as pd, joblib, requests
from IPython.display import display
from werkzeug.serving import make_server
from sandbox.target_app import app
from src.authorization.authorize import authorize
from src.recon.recon import discover, build_session, InjectionPoint
from src.intelligence.select import select_all
from src.governance import govern
from src.execution import baseline
from src.detection import detect
from src.detection.oracles import _not_demonstrable
from src.agent import build_agent, AuditLog

# launch the self-owned sandbox on an allowlisted port
_srv = make_server("127.0.0.1", 5000, app)
threading.Thread(target=_srv.serve_forever, daemon=True).start()
time.sleep(0.5)
SANDBOX = "http://127.0.0.1:5000"
POINTS = discover(SANDBOX, "pyapp")
SESS = build_session(SANDBOX, "pyapp")

RESULTS = []
def expect(question, condition, detail=""):
    ok = bool(condition)
    RESULTS.append({"question": question, "result": "PASS" if ok else "FAIL"})
    print(("PASS   " if ok else "FAIL   ") + question + ("" if ok else f"   <- {detail}"))
    return ok

print("setup ok | sandbox reachable:", requests.get(SANDBOX, timeout=2).status_code == 200,
      "| injection points:", len(POINTS))

127.0.0.1 - - [20/Jul/2026 17:23:59] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /sqli?user=admin HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /xss?name=friend HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /comment HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /exec HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /account HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET /fetch?url=http://example.com HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:23:59] "GET / HTTP/1.1" 200 -


setup ok | sandbox reachable: True | injection points: 6


## 1. Authorization — *Can the tool be pointed at something that isn't ours?*

In [2]:
expect("Q1  the scope firewall refuses an out-of-scope third-party site",
       not authorize("http://example.com").approved)

PASS   Q1  the scope firewall refuses an out-of-scope third-party site


True

## 2. Selection — *Do we test broadly, or collapse to a single technique?*

In [3]:
sel = select_all([InjectionPoint("sqli", f"{SANDBOX}/sqli", "GET", "q", "search_field", ["sqli"])],
                 k_per_type=2)
techniques = {p["type"] for _, ps in sel for p in ps}
expect("Q2  selection stratifies across multiple attack techniques",
       len(techniques) >= 3, f"only got {techniques}")

PASS   Q2  selection stratifies across multiple attack techniques


True

## 3. Governance — *Are dangerous payloads recorded, even when fired autonomously?*

In [4]:
class _Pt: name = "pt"; full_url = "u"
destructive = {"id": "DROP", "attack_class": "sqli", "type": "stacked",
               "is_destructive": True, "payload": "'; DROP TABLE users--"}
gate = govern([(_Pt(), [destructive])])
expect("Q3  destructive payloads are flagged in the audit (visible, not hidden)",
       len(gate.flagged) == 1)

PASS   Q3  destructive payloads are flagged in the audit (visible, not hidden)


True

## 4. Detection oracles — *Do we ever call an unconfirmed result 'safe'?*

In [5]:
conf = _not_demonstrable("out_of_band", "a callback server")
expect("Q4  an undemonstrable oracle returns None (unproven), never False (safe)",
       conf.confirmed is None)

PASS   Q4  an undemonstrable oracle returns None (unproven), never False (safe)


True

## 5. Execution + detection — *Do we actually confirm a REAL exploit on the live target?*

In [6]:
sqli_pt = next(p for p in POINTS if p.name == "sqli")
base = baseline(SESS, sqli_pt)
conf, _ = detect(SESS, sqli_pt, {"payload": "' OR '1'='1", "oracle": "differential"}, base)
expect("Q5  a real SQL injection is confirmed via the differential oracle",
       conf.confirmed is True)

127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user=test HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+OR+'1'%3D'1 HTTP/1.1" 200 -


127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+OR+'1'%3D'2 HTTP/1.1" 200 -


PASS   Q5  a real SQL injection is confirmed via the differential oracle


True

## 6. Audit — *Can the record of what the agent did be secretly altered?*

In [7]:
tf = Path(tempfile.mkdtemp()) / "audit.jsonl"
log = AuditLog(tf); log.record("start", {"g": "x"}); log.record("act", {"t": "recon"})
lines = tf.read_text(encoding="utf-8").splitlines()
rec = json.loads(lines[0]); rec["data"]["g"] = "EVIL"; lines[0] = json.dumps(rec)   # tamper
tf.write_text(chr(10).join(lines) + chr(10), encoding="utf-8")
ok, _ = AuditLog.verify(tf)
expect("Q6  tampering with the audit ledger is detected", not ok)

PASS   Q6  tampering with the audit ledger is detected


True

## 7. Agent (end-to-end) — *Does the whole agent actually work, start to finish?*

In [8]:
state = build_agent(audit=AuditLog(Path(tempfile.mkdtemp()) / "a.jsonl")).invoke({"target": SANDBOX})
expect("Q7  the full LangGraph agent runs end-to-end and confirms at least one exploit",
       state["status"] == "done" and any(f["confirmed"] is True for f in state["findings"]))

127.0.0.1 - - [20/Jul/2026 17:24:00] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user=admin HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /xss?name=friend HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /comment HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /exec HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /account HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /fetch?url=http://example.com HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user=test HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+OR+1%3D1%23 HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+OR+1%3D2%23 HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+OR+SUBSTRING(user(),1,1)%3D'a'/* HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+AND+SUBSTRING(user(),1,1)%3D'a'/* HTTP/1.1" 200 -
127.0.0.1 - - [20/Jul/2026 17:24:00] "GET /sqli?user='+

PASS   Q7  the full LangGraph agent runs end-to-end and confirms at least one exploit


True

## 8. Model — *Where does the attack classifier break?*

In [9]:
clf = joblib.load(ROOT / "models" / "clf_binary.pkl")
attacks = ["<script>alert(1)</script>", "' OR '1'='1", "<img src=x onerror=alert(1)>"]
evaded = sum(1 for a in attacks if clf.predict([urllib.parse.quote(a)])[0] == "benign")
expect("Q8  the model is evadable by URL-encoding the same attacks (a real weakness)",
       evaded >= 1, f"evaded {evaded}/{len(attacks)}")

PASS   Q8  the model is evadable by URL-encoding the same attacks (a real weakness)


True

## Summary

In [10]:
df = pd.DataFrame(RESULTS)
passed = int((df["result"] == "PASS").sum())
print("=" * 56)
print(f"{passed}/{len(df)} key questions PASSED")
print("=" * 56)
display(df)
_srv.shutdown()
print("sandbox stopped.")

8/8 key questions PASSED


,question,result
0,Q1 the scope firewall refuses an out-of-scope...,PASS
1,Q2 selection stratifies across multiple attac...,PASS
2,Q3 destructive payloads are flagged in the au...,PASS
3,Q4 an undemonstrable oracle returns None (unp...,PASS
4,Q5 a real SQL injection is confirmed via the ...,PASS
5,Q6 tampering with the audit ledger is detected,PASS
6,Q7 the full LangGraph agent runs end-to-end a...,PASS
7,Q8 the model is evadable by URL-encoding the ...,PASS


sandbox stopped.
